# SEO Tools 
## Scraping Metadata 

In [2]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from bs4 import BeautifulSoup

#Define headers to mimic a real browser since I keep getting a 403 error
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.google.com/",
    "Accept-Language": "en-US,en;q=0.9"
}

#Get sitemap (XML)
sitemap_url = "https://www.schmittchevrolet.com/page-sitemap.xml"
response = requests.get(sitemap_url, headers=headers)

if response.status_code == 200:
    root = ET.fromstring(response.text)

    #Extract all URLs from sitemap
    namespace = {"ns": "http://www.sitemaps.org/schemas/sitemap/0.9"}
    urls = [elem.text for elem in root.findall(".//ns:loc", namespace)]

    print(f"Found {len(urls)} URLs in the sitemap.\n")

    #Scrape each page for metadata
    data = []
    for url in urls:
        try:
            page_response = requests.get(url, headers=headers, timeout=10)
            if page_response.status_code == 200:
                soup = BeautifulSoup(page_response.text, "html.parser")

                #Extract metadata
                title = soup.title.text.strip() if soup.title else None
                description = (
                    soup.find("meta", attrs={"name": "description"})["content"].strip()
                    if soup.find("meta", attrs={"name": "description"})
                    else None
                )
                keywords = (
                    soup.find("meta", attrs={"name": "keywords"})["content"].strip()
                    if soup.find("meta", attrs={"name": "keywords"})
                    else None
                )

                #Determine if each metadata element is missing (1 = missing, 0 = present)
                missing_title = 1 if title is None else 0
                missing_description = 1 if description is None else 0
                missing_keywords = 1 if keywords is None else 0

                title = title if title else "No Title"
                description = description if description else "No Description"
                keywords = keywords if keywords else "No Keywords"

                data.append([url, title, description, keywords, missing_title, missing_description, missing_keywords])

            else:
                print(f"Failed to scrape {url}: {page_response.status_code}")

        except Exception as e:
            print(f"Error scraping {url}: {e}")

    df = pd.DataFrame(data, columns=["URL", "Title", "Description", "Keywords", "Missing_Title", "Missing_Description", "Missing_Keywords"])

    df.to_csv("weber_sitemap_metadata.csv", index=False, encoding="utf-8")

    print("\n Data saved to sitemap_metadata.csv")

else:
    print(f"Error {response.status_code}: Unable to access sitemap.")


Found 193 URLs in the sitemap.

Error scraping https://www.schmittchevrolet.com/about-us/: Response ended prematurely
Error scraping https://www.schmittchevrolet.com/careers/employment-application/: Response ended prematurely
Error scraping https://www.schmittchevrolet.com/finance/apply-for-financing/: Response ended prematurely
Error scraping https://www.schmittchevrolet.com/service/contact-service/: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error scraping https://www.schmittchevrolet.com/vehicle-finder-service/: Response ended prematurely
Error scraping https://www.schmittchevrolet.com/service/: Response ended prematurely
Error scraping https://www.schmittchevrolet.com/careers/: Response ended prematurely
Error scraping https://www.schmittchevrolet.com/vehicle-configurator/: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Error scraping https://www.schmittchevrolet.com/complyauto-request-p

In [3]:
import time
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from bs4 import BeautifulSoup
from requests.exceptions import ChunkedEncodingError, ConnectionError

# Headers to mimic a real browser but force uncompressed response
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.google.com/",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Encoding": "identity",  # key: no gzip/br
}

sitemap_url = "https://www.schmittchevrolet.com/page-sitemap.xml"

session = requests.Session()
session.headers.update(headers)


def fetch_with_retry(url, max_tries=3, sleep_secs=2):
    """Fetch a URL with a few retries to handle 'Response ended prematurely' etc."""
    last_err = None
    for attempt in range(1, max_tries + 1):
        try:
            resp = session.get(url, timeout=15)
            resp.raise_for_status()
            # Force load full content so chunked/partial responses explode here
            _ = resp.content
            return resp
        except (ChunkedEncodingError, ConnectionError) as e:
            last_err = e
            print(f"Attempt {attempt} for {url} failed: {e}")
            if attempt < max_tries:
                time.sleep(sleep_secs)
        except Exception as e:
            # Other errors that we do not expect to be transient
            print(f"Non-retryable error fetching {url}: {e}")
            return None
    print(f"Giving up on {url} after {max_tries} attempts. Last error: {last_err}")
    return None


# 1. Get sitemap
resp = fetch_with_retry(sitemap_url, max_tries=2)
if not resp:
    raise SystemExit("Could not fetch sitemap at all.")

try:
    root = ET.fromstring(resp.content)
except ET.ParseError as e:
    print("Sitemap contents (first 500 chars):")
    print(resp.text[:500])
    raise SystemExit(f"Error parsing sitemap XML: {e}")

# 2. Extract URLs from sitemap ignoring exact namespace
urls = [elem.text for elem in root.iter("{*}loc") if elem.text]
print(f"Found {len(urls)} URLs in the sitemap.\n")

data = []

# 3. Scrape each page
for i, url in enumerate(urls, start=1):
    print(f"[{i}/{len(urls)}] Scraping {url}")
    page_resp = fetch_with_retry(url, max_tries=3, sleep_secs=2)
    if not page_resp:
        continue  # already logged as failure

    soup = BeautifulSoup(page_resp.text, "html.parser")

    # Title
    title_tag = soup.title
    title = title_tag.text.strip() if title_tag and title_tag.text else None

    # Meta description
    desc_tag = soup.find("meta", attrs={"name": "description"})
    description = desc_tag.get("content", "").strip() if desc_tag and desc_tag.get("content") else None

    # Meta keywords
    key_tag = soup.find("meta", attrs={"name": "keywords"})
    keywords = key_tag.get("content", "").strip() if key_tag and key_tag.get("content") else None

    missing_title = 1 if not title else 0
    missing_description = 1 if not description else 0
    missing_keywords = 1 if not keywords else 0

    title = title if title else "No Title"
    description = description if description else "No Description"
    keywords = keywords if keywords else "No Keywords"

    data.append(
        [
            url,
            title,
            description,
            keywords,
            missing_title,
            missing_description,
            missing_keywords,
        ]
    )

# 4. Save to CSV
if data:
    df = pd.DataFrame(
        data,
        columns=[
            "URL",
            "Title",
            "Description",
            "Keywords",
            "Missing_Title",
            "Missing_Description",
            "Missing_Keywords",
        ],
    )
    out_name = "schmitt_sitemap_metadata.csv"
    df.to_csv(out_name, index=False, encoding="utf-8")
    print(f"\nData saved to {out_name}")
else:
    print("No pages successfully scraped. Server is probably being a pain.")


Found 0 URLs in the sitemap.

No pages successfully scraped. Server is probably being a pain.


In [ ]:
import time
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from bs4 import BeautifulSoup
from requests.exceptions import ChunkedEncodingError, ConnectionError, HTTPError

SITEMAP_URL = "https://www.webergranitecitychevy.com/page-sitemap.xml"
OUTPUT_CSV = "weber_sitemap_metadata.csv"

# Headers to mimic a real browser and force uncompressed response
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Referer": "https://www.google.com/",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Encoding": "identity",  # avoid gzip related chunk issues
}


def local_name(tag: str) -> str:
    """
    Strip XML namespace from a tag.
    Example: "{http://example.com}loc" -> "loc"
             "loc" -> "loc"
    """
    return tag.split("}")[-1] if "}" in tag else tag


def fetch_with_retry(session: requests.Session, url: str, max_tries: int = 3, sleep_secs: int = 2):
    """
    Fetch a URL with retries on transient errors like chunked encoding failures
    and connection resets. Returns a Response or None if all attempts fail.
    """
    last_err = None
    for attempt in range(1, max_tries + 1):
        try:
            resp = session.get(url, timeout=20)
            # Raise for 4xx or 5xx to handle bad responses cleanly
            resp.raise_for_status()
            # Force reading full content to trigger chunk errors here
            _ = resp.content
            return resp
        except (ChunkedEncodingError, ConnectionError) as e:
            last_err = e
            print(f"Attempt {attempt} for {url} failed (connection issue): {e}")
            if attempt < max_tries:
                time.sleep(sleep_secs)
        except HTTPError as e:
            print(f"HTTP error for {url}: {e}")
            return None
        except Exception as e:
            # Any other unexpected error, do not retry forever
            last_err = e
            print(f"Non retryable error for {url}: {e}")
            break

    print(f"Giving up on {url} after {max_tries} attempts. Last error: {last_err}")
    return None


def parse_sitemap_xml(content: bytes) -> ET.Element:
    """
    Parse sitemap XML and return the root element.
    Raises SystemExit with a helpful message on parse failure.
    """
    try:
        root = ET.fromstring(content)
        return root
    except ET.ParseError as e:
        print("Failed to parse sitemap XML. First 500 characters below:")
        try:
            text_preview = content.decode("utf-8", errors="replace")[:500]
        except Exception:
            text_preview = str(content[:500])
        print(text_preview)
        raise SystemExit(f"Error parsing sitemap XML: {e}")


def extract_urls_from_urlset(root: ET.Element):
    """
    Given a <urlset> sitemap root, extract all <loc> entries.
    Handles namespaces by stripping them.
    """
    urls = [
        elem.text.strip()
        for elem in root.iter()
        if local_name(elem.tag) == "loc" and elem.text
    ]
    return urls


def extract_sitemap_urls_from_index(root: ET.Element):
    """
    Given a <sitemapindex> sitemap root, extract all child sitemap <loc> URLs.
    These are URLs pointing to other sitemap XML files.
    """
    sitemap_urls = [
        elem.text.strip()
        for elem in root.iter()
        if local_name(elem.tag) == "loc" and elem.text
    ]
    return sitemap_urls


def collect_all_page_urls(session: requests.Session, sitemap_url: str) -> list:
    """
    Entry point:
    1. Fetch the top level sitemap URL
    2. Determine if it is a sitemapindex or urlset
    3. If sitemapindex, fetch each subsitemap and collect urls
    4. If urlset, just collect urls directly
    """
    print(f"Fetching sitemap: {sitemap_url}")
    resp = fetch_with_retry(session, sitemap_url, max_tries=3, sleep_secs=2)
    if not resp:
        raise SystemExit(f"Could not fetch sitemap at {sitemap_url}")

    root = parse_sitemap_xml(resp.content)
    root_name = local_name(root.tag)
    print(f"Sitemap root tag: {root_name}")

    all_urls = []

    if root_name == "urlset":
        # Simple sitemap with direct URLs
        urls = extract_urls_from_urlset(root)
        print(f"Found {len(urls)} URLs in urlset.")
        all_urls.extend(urls)

    elif root_name == "sitemapindex":
        # Sitemap index, needs second level fetches
        sitemap_urls = extract_sitemap_urls_from_index(root)
        print(f"Found {len(sitemap_urls)} child sitemaps in sitemapindex.")

        for i, sm_url in enumerate(sitemap_urls, start=1):
            print(f"[{i}/{len(sitemap_urls)}] Fetching child sitemap: {sm_url}")
            sm_resp = fetch_with_retry(session, sm_url, max_tries=3, sleep_secs=2)
            if not sm_resp:
                continue
            sm_root = parse_sitemap_xml(sm_resp.content)
            sm_root_name = local_name(sm_root.tag)
            if sm_root_name != "urlset":
                print(f"Warning: Child sitemap {sm_url} root is {sm_root_name}, expected urlset. Skipping.")
                continue
            urls = extract_urls_from_urlset(sm_root)
            print(f"  Found {len(urls)} URLs.")
            all_urls.extend(urls)
    else:
        raise SystemExit(f"Unknown sitemap root tag: {root_name}")

    # Deduplicate while preserving order
    seen = set()
    unique_urls = []
    for u in all_urls:
        if u not in seen:
            seen.add(u)
            unique_urls.append(u)

    print(f"Total unique URLs collected: {len(unique_urls)}")
    return unique_urls


def scrape_metadata_for_urls(session: requests.Session, urls: list) -> pd.DataFrame:
    """
    For each URL, fetch the page and extract title, meta description, and meta keywords.
    Returns a pandas DataFrame with metadata and missing flags.
    """
    records = []

    for i, url in enumerate(urls, start=1):
        print(f"[{i}/{len(urls)}] Scraping {url}")
        resp = fetch_with_retry(session, url, max_tries=3, sleep_secs=2)
        if not resp:
            continue

        soup = BeautifulSoup(resp.text, "html.parser")

        # Title
        title_tag = soup.title
        title = title_tag.text.strip() if title_tag and title_tag.text else None

        # Meta description
        desc_tag = soup.find("meta", attrs={"name": "description"})
        description = (
            desc_tag.get("content", "").strip()
            if desc_tag and desc_tag.get("content")
            else None
        )

        # Meta keywords
        key_tag = soup.find("meta", attrs={"name": "keywords"})
        keywords = (
            key_tag.get("content", "").strip()
            if key_tag and key_tag.get("content")
            else None
        )

        missing_title = 0 if title else 1
        missing_description = 0 if description else 1
        missing_keywords = 0 if keywords else 1

        title = title if title else "No Title"
        description = description if description else "No Description"
        keywords = keywords if keywords else "No Keywords"

        records.append(
            {
                "URL": url,
                "Title": title,
                "Description": description,
                "Keywords": keywords,
                "Missing_Title": missing_title,
                "Missing_Description": missing_description,
                "Missing_Keywords": missing_keywords,
            }
        )

    if not records:
        print("Warning: No pages were successfully scraped.")
        return pd.DataFrame(
            columns=[
                "URL",
                "Title",
                "Description",
                "Keywords",
                "Missing_Title",
                "Missing_Description",
                "Missing_Keywords",
            ]
        )

    df = pd.DataFrame(records)
    return df


def main():
    session = requests.Session()
    session.headers.update(HEADERS)

    # 1. Collect all URLs from sitemap or sitemapindex
    urls = collect_all_page_urls(session, SITEMAP_URL)
    if not urls:
        print("No URLs found in sitemap. Exiting.")
        return

    # 2. Scrape metadata for each URL
    df = scrape_metadata_for_urls(session, urls)

    # 3. Save to CSV
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"\nSaved metadata for {len(df)} pages to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


Fetching sitemap: https://www.webergranitecitychevy.com/page-sitemap.xml
Attempt 1 for https://www.webergranitecitychevy.com/page-sitemap.xml failed (connection issue): Response ended prematurely
Sitemap root tag: urlset
Found 250 URLs in urlset.
Total unique URLs collected: 250
[1/250] Scraping https://www.webergranitecitychevy.com/
Attempt 1 for https://www.webergranitecitychevy.com/ failed (connection issue): ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
[2/250] Scraping https://www.webergranitecitychevy.com/careers/employment-application/
[3/250] Scraping https://www.webergranitecitychevy.com/blog-posts-sitemap/
[4/250] Scraping https://www.webergranitecitychevy.com/models-sitemap/
[5/250] Scraping https://www.webergranitecitychevy.com/comparisons-sitemap/
[6/250] Scraping https://www.webergranitecitychevy.com/about-us/customer-testimonials/
[7/250] Scraping https://www.webergranitecitychevy.com/service/contact-service/
[8/250] Scrapin